# Sector Analysis

Comparing performance across Technology, Financials, and ETF categories.

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load data
prices = pd.read_csv("../data/processed/market_prices.csv", parse_dates=["date"])
metrics = pd.read_csv("../data/processed/analytics_metrics.csv", parse_dates=["date"])

# Merge category into metrics so we can group by it
metrics = metrics.merge(prices[["date", "ticker", "category"]].drop_duplicates(), on=["date", "ticker"], how="left")

print("market_prices shape  :", prices.shape)
print("analytics_metrics shape:", metrics.shape)
print("\nCategories:", metrics["category"].unique())

## 1. Cumulative Return by Category

Average cumulative return over time for each sector category.

In [ ]:
plt.rcParams["figure.figsize"] = (14, 5)

# Group by category + date, take the mean cumulative return
cum_ret = (
    metrics.groupby(["category", "date"])["cumulative_return"]
    .mean()
    .reset_index()
)

category_colors = {"Technology": "steelblue", "Financials": "seagreen", "ETF": "orange"}

fig, ax = plt.subplots()

for cat, group in cum_ret.groupby("category"):
    group = group.sort_values("date")
    ax.plot(group["date"], group["cumulative_return"], label=cat, color=category_colors.get(cat))

ax.set_title("Average Cumulative Return by Category Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Avg Cumulative Return")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend(title="Category")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Average Daily Return by Category (Bar Chart)

Which category had the best average daily return on the most recent date in the dataset?

In [ ]:
latest_date = metrics["date"].max()
latest = metrics[metrics["date"] == latest_date]

avg_daily = latest.groupby("category")["daily_return"].mean().sort_values()

fig, ax = plt.subplots()

bars = ax.bar(
    avg_daily.index,
    avg_daily.values,
    color=[category_colors.get(c, "gray") for c in avg_daily.index],
    width=0.5
)

ax.set_title(f"Average Daily Return by Category  (as of {latest_date.date()})")
ax.set_xlabel("Category")
ax.set_ylabel("Avg Daily Return")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

# Label each bar with its value
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.0001,
        f"{height:.4f}",
        ha="center", va="bottom", fontsize=10
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Average 30-Day Volatility by Category Over Time

Rolling 30-day volatility shows how much price swings within each sector change through time.

In [ ]:
vol_30 = (
    metrics.groupby(["category", "date"])["rolling_vol_30"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots()

for cat, group in vol_30.groupby("category"):
    group = group.sort_values("date")
    ax.plot(group["date"], group["rolling_vol_30"], label=cat, color=category_colors.get(cat))

ax.set_title("Average 30-Day Rolling Volatility by Category Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Avg Rolling Volatility (30-Day)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend(title="Category")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Total Return per Ticker grouped by Category

Each bar is one ticker. Color indicates its category. Sorted from highest to lowest return.

In [ ]:
# Get each ticker's cumulative return and category on the latest date
ticker_ret = (
    latest[["ticker", "category", "cumulative_return"]]
    .drop_duplicates(subset=["ticker"])
    .sort_values("cumulative_return", ascending=True)   # ascending so highest is at top of hbar
)

colors = [category_colors.get(c, "gray") for c in ticker_ret["category"]]

fig, ax = plt.subplots(figsize=(14, max(5, len(ticker_ret) * 0.4)))

bars = ax.barh(ticker_ret["ticker"], ticker_ret["cumulative_return"], color=colors)

ax.set_title(f"Cumulative Return per Ticker by Category  (as of {latest_date.date()})")
ax.set_xlabel("Cumulative Return")
ax.set_ylabel("Ticker")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")

# Add a simple legend for categories
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in category_colors.items()]
ax.legend(handles=legend_elements, title="Category", loc="lower right")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Volatility vs Return (Scatter)

Each point is one ticker on the latest date. The x-axis shows 90-day rolling volatility (risk) and the y-axis shows cumulative return (reward). This is a classic risk-return view.

In [ ]:
scatter_data = (
    latest[["ticker", "category", "rolling_vol_90", "cumulative_return"]]
    .drop_duplicates(subset=["ticker"])
    .dropna(subset=["rolling_vol_90", "cumulative_return"])
)

fig, ax = plt.subplots()

for cat, group in scatter_data.groupby("category"):
    ax.scatter(
        group["rolling_vol_90"],
        group["cumulative_return"],
        color=category_colors.get(cat, "gray"),
        label=cat,
        s=80,
        zorder=3
    )
    # Annotate each point with the ticker name
    for _, row in group.iterrows():
        ax.annotate(
            row["ticker"],
            xy=(row["rolling_vol_90"], row["cumulative_return"]),
            xytext=(5, 3),
            textcoords="offset points",
            fontsize=8,
            color="dimgray"
        )

ax.set_title(f"90-Day Volatility vs Cumulative Return  (as of {latest_date.date()})")
ax.set_xlabel("Rolling Volatility (90-Day)")
ax.set_ylabel("Cumulative Return")
ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.legend(title="Category")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()